# Coffee Quality Datatbase from CQI Wrangled

# Table of Contents — Top Rated Coffee Dataset Wrangled

1. Imports & Pathways    
2. Data Inspection  
3. Data Wrangle – Harvest Year 
4. Data Wrangle – Origin
5. Reorganized the Columns
6. Export  

## 1. Imports/Pathways

In [1]:
# Import libraries

import pandas as pd
import numpy as np
import os
import re

In [2]:
# File Pathway
path = r"C:\Users\Chase\anaconda_projects\Exercise_6_Coffee\A6_Coffee"

In [3]:
# Import dataset
df_cqi_wrangle = pd.read_csv(
    r'C:\Users\Chase\anaconda_projects\Exercise_6_Coffee\A6_Coffee\02_Data\Prepared_Data\Coffee_Quality_database_from_CQI\coffee_quality_cqi_clean.csv')

## 2. Data Inspection

In [4]:
df_cqi_wrangle.head()

,species,owner,origin_country,region,local_partner,harvest_year,variety,processing_method,aroma,flavor,...,body,balance,uniformity,clean_cup,sweetness,cupper_points,total_cup_points,moisture,unit_of_measurement,altitude_mean_meters
0,arabica,metad plc,ethiopia,guji-hambela,metad agricultural development plc,2014,unknown,washed / wet,8.67,8.83,...,8.50,8.42,10.0,10.0,10.0,8.75,90.58,0.12,m,2075.0
1,arabica,metad plc,ethiopia,guji-hambela,metad agricultural development plc,2014,other,washed / wet,8.75,8.67,...,8.42,8.42,10.0,10.0,10.0,8.58,89.92,0.12,m,2075.0
2,arabica,grounds for health admin,guatemala,unknown,specialty coffee association,NaN,bourbon,unknown,8.42,8.50,...,8.33,8.42,10.0,10.0,10.0,9.25,89.75,0.00,m,1700.0
3,arabica,yidnekachew dabessa,ethiopia,oromia,metad agricultural development plc,2014,unknown,natural / dry,8.17,8.58,...,8.50,8.25,10.0,10.0,10.0,8.67,89.00,0.11,m,2000.0
4,arabica,metad plc,ethiopia,guji-hambela,metad agricultural development plc,2014,other,washed / wet,8.25,8.50,...,8.42,8.33,10.0,10.0,10.0,8.58,88.83,0.12,m,2075.0


In [ ]:
# Header check
df_cqi_wrangle.columns.tolist()

## 3. Data Wrangle - Harvest Year

In [6]:
# Checking the unique list of harvest_years
unique_raw_inputs = df_cqi_wrangle['harvest_year'].dropna().astype(str).str.strip().unique()
unique_raw_inputs.sort()
for entry in unique_raw_inputs:
    print(entry)

08/09 crop
1t/2011
2009 - 2010
2009 / 2010
2009-2010
2009/2010
2010
2010-2011
2011
2011/2012
2012
2013
2013/2014
2014
2014/2015
2015
2015/2016
2016
2016 / 2017
2016/2017
2017
2017 / 2018
2018
23-jul-10
3t/2011
47/2010
4t/10
4t/2010
4t/2011
4t72010
abril - julio
abril - julio /2011
august to december
december 2009-march 2010
fall 2009
jan-11
january through april
mar-10
may-august
mayo a julio
mmm
sept 2009 - april 2010
spring 2011 in colombia.
test


In [7]:
# Create a column called harvest_year_raw to work with
df_cqi_wrangle['harvest_year_raw'] = df_cqi_wrangle['harvest_year'].astype(str).str.strip()

In [8]:
# Extract valid years, 4 digit
df_cqi_wrangle['harvest_year_extracted'] = df_cqi_wrangle['harvest_year_raw'].apply(
    lambda x: re.findall(r'\b(?:19|20)\d{2}\b', x))

In [9]:
# Work check
df_cqi_wrangle[['harvest_year_raw', 'harvest_year_extracted']].head(20)

,harvest_year_raw,harvest_year_extracted
0,2014,[2014]
1,2014,[2014]
2,nan,[]
3,2014,[2014]
4,2014,[2014]
5,2013,[2013]
6,2012,[2012]
7,mar-10,[]
8,mar-10,[]
9,2014,[2014]


In [10]:
# Flag entries with seasonal or junk to work on
def flag_harvest_year(row):
    raw = str(row['harvest_year_raw']).lower()
    extracted = row['harvest_year_extracted']
    
    if extracted:
        if any(season in raw for season in ['spring', 'summer', 'fall', 'winter', '1t', '2t', '3t', '4t']):
            return 'seasonal'
        else:
            return 'valid'
    elif raw in ['n/a', 'unknown', 'test', 'mmm', 'nan']:
        return 'junk'
    else:
        return 'ambiguous'

df_cqi_wrangle['harvest_year_flag'] = df_cqi_wrangle.apply(flag_harvest_year, axis=1)

In [11]:
#Work check
df_cqi_wrangle[['harvest_year_raw', 'harvest_year_extracted', 'harvest_year_flag']].head(20)

,harvest_year_raw,harvest_year_extracted,harvest_year_flag
0,2014,[2014],valid
1,2014,[2014],valid
2,nan,[],junk
3,2014,[2014],valid
4,2014,[2014],valid
5,2013,[2013],valid
6,2012,[2012],valid
7,mar-10,[],ambiguous
8,mar-10,[],ambiguous
9,2014,[2014],valid


In [12]:
# Value count of the flags
df_cqi_wrangle['harvest_year_flag'].value_counts()

harvest_year_flag
valid        1259
junk           49
ambiguous      21
seasonal       10
Name: count, dtype: int64

In [13]:
# Make a list of seasonal and ambiguous
df_needs_work = df_cqi_wrangle[df_cqi_wrangle['harvest_year_flag'].isin(['seasonal', 'ambiguous'])]

In [14]:
# Work check
df_needs_work[['harvest_year_raw', 'harvest_year_extracted', 'harvest_year_flag']].head(20)

,harvest_year_raw,harvest_year_extracted,harvest_year_flag
7,mar-10,[],ambiguous
8,mar-10,[],ambiguous
14,mar-10,[],ambiguous
16,may-august,[],ambiguous
51,fall 2009,[2009],seasonal
311,jan-11,[],ambiguous
312,4t/10,[],ambiguous
374,23-jul-10,[],ambiguous
395,january through april,[],ambiguous
437,1t/2011,[2011],seasonal


In [16]:
# Filter unresolved rows
df_unresolved = df_cqi_wrangle[
    df_cqi_wrangle['harvest_year_flag'].isin(['ambiguous', 'junk']) &
    df_cqi_wrangle['harvest_year_extracted'].apply(lambda x: len(x) == 0)
]

In [20]:
# Review the unresolved rows
with pd.option_context('display.max_rows', None):
    display(df_unresolved[['harvest_year_raw', 'harvest_year_flag']])

,harvest_year_raw,harvest_year_flag
2,nan,junk
7,mar-10,ambiguous
8,mar-10,ambiguous
14,mar-10,ambiguous
16,may-august,ambiguous
30,nan,junk
95,nan,junk
104,nan,junk
127,nan,junk
142,nan,junk


Looking at the list, using Copilot, we created a clean block patch to recover certain entries with years hidden in it.

In [22]:
# Select and move over the remaining years within the unresolved rows
manual_years = {
    'mar-10': [2010],
    'jan-11': [2011],
    '4t/10': [2010],
    '23-jul-10': [2010]
}

for raw, year_list in manual_years.items():
    mask = (
        (df_cqi_wrangle['harvest_year_raw'] == raw) &
        (df_cqi_wrangle['harvest_year_extracted'].apply(lambda x: len(x) == 0))
    )
    df_cqi_wrangle.loc[mask, 'harvest_year_extracted'] = df_cqi_wrangle.loc[mask, 'harvest_year_extracted'].apply(
        lambda x: year_list
    )

In [24]:
# Work check
df_unresolved_final = df_cqi_wrangle[
    df_cqi_wrangle['harvest_year_extracted'].apply(lambda x: isinstance(x, list) and len(x) == 0)
]

# Display the result
df_unresolved_final[['harvest_year_raw', 'harvest_year_flag']]

,harvest_year_raw,harvest_year_flag
2,nan,junk
16,may-august,ambiguous
30,nan,junk
95,nan,junk
104,nan,junk
127,nan,junk
142,nan,junk
144,nan,junk
169,mmm,junk
170,test,junk


2 more entries that can be extracted. 08/09 crop into 2008.

In [25]:
# Extract the last 2 rows that have vaild year
manual_years = {
    '08/09 crop': [2008]  # Only the first year
}

for raw, year_list in manual_years.items():
    mask = (
        (df_cqi_wrangle['harvest_year_raw'] == raw) &
        (df_cqi_wrangle['harvest_year_extracted'].apply(lambda x: isinstance(x, list) and len(x) == 0))
    )
    df_cqi_wrangle.loc[mask, 'harvest_year_extracted'] = df_cqi_wrangle.loc[mask, 'harvest_year_extracted'].apply(
        lambda x: year_list
    )

In [26]:
# Filter for unresolved rows
df_unresolved_final = df_cqi_wrangle[
    df_cqi_wrangle['harvest_year_flag'].isin(['ambiguous', 'junk']) &
    df_cqi_wrangle['harvest_year_extracted'].apply(lambda x: isinstance(x, list) and len(x) == 0)
]

In [27]:
# Work check, display results
with pd.option_context('display.max_rows', None):
    display(df_unresolved_final[['harvest_year_raw', 'harvest_year_flag']])

,harvest_year_raw,harvest_year_flag
2,nan,junk
16,may-august,ambiguous
30,nan,junk
95,nan,junk
104,nan,junk
127,nan,junk
142,nan,junk
144,nan,junk
169,mmm,junk
170,test,junk


In [28]:
# Replace empty spaces in harvest_year_extracted with NaN to finish cleaning the rows.
df_cqi_wrangle['harvest_year_extracted'] = df_cqi_wrangle['harvest_year_extracted'].apply(
    lambda x: np.nan if isinstance(x, list) and len(x) == 0 else x
)

In [29]:
# Work check
df_cqi_wrangle['harvest_year_extracted'].value_counts(dropna=False)

harvest_year_extracted
[2012]          354
[2014]          233
[2013]          181
[2015]          129
[2016]          124
[2017]           70
NaN              58
[2011]           32
[2013, 2014]     29
[2015, 2016]     28
[2017, 2018]     19
[2009, 2010]     19
[2014, 2015]     19
[2010]           15
[2010]            8
[2016, 2017]      7
[2010, 2011]      6
[2011, 2012]      2
[2008]            2
[2011]            2
[2009]            1
[2018]            1
Name: count, dtype: int64

Need to clean up the double years.

In [30]:
# Cleaning up the double year entries
df_cqi_wrangle['harvest_year_extracted'] = df_cqi_wrangle['harvest_year_extracted'].apply(
    lambda x: [x[0]] if isinstance(x, list) and len(x) > 1 else x
)

In [31]:
# Work check
df_cqi_wrangle['harvest_year_extracted'].value_counts(dropna=False)

harvest_year_extracted
[2012]    354
[2014]    252
[2013]    210
[2015]    157
[2016]    131
[2017]     89
NaN        58
[2011]     34
[2010]     21
[2009]     20
[2010]      8
[2011]      2
[2008]      2
[2018]      1
Name: count, dtype: int64

In [32]:
# Checking headers to see what needs to be dropped
df_cqi_wrangle.columns.tolist()

['species',
 'owner',
 'origin_country',
 'region',
 'local_partner',
 'harvest_year',
 'variety',
 'processing_method',
 'aroma',
 'flavor',
 'aftertaste',
 'acidity',
 'body',
 'balance',
 'uniformity',
 'clean_cup',
 'sweetness',
 'cupper_points',
 'total_cup_points',
 'moisture',
 'unit_of_measurement',
 'altitude_mean_meters',
 'harvest_year_raw',
 'harvest_year_extracted',
 'harvest_year_flag']

In [34]:
# Double check harevest_year_extracted is the correct and clean cloumn we need
df_cqi_wrangle['harvest_year_extracted'].value_counts(dropna=False)

harvest_year_extracted
[2012]    354
[2014]    252
[2013]    210
[2015]    157
[2016]    131
[2017]     89
NaN        58
[2011]     34
[2010]     21
[2009]     20
[2010]      8
[2011]      2
[2008]      2
[2018]      1
Name: count, dtype: int64

Looks good.

In [35]:
# Drop columns
df_cqi_wrangle.drop(columns=[
    'harvest_year',
    'harvest_year_raw',     
    'harvest_year_flag' 
], inplace=True)

In [37]:
# Rename harvest_year_extracted to just harvest_year.
df_cqi_wrangle.rename(columns={'harvest_year_extracted': 'harvest_year'}, inplace=True)

In [38]:
# Work check on headers
df_cqi_wrangle.columns.tolist()

['species',
 'owner',
 'origin_country',
 'region',
 'local_partner',
 'variety',
 'processing_method',
 'aroma',
 'flavor',
 'aftertaste',
 'acidity',
 'body',
 'balance',
 'uniformity',
 'clean_cup',
 'sweetness',
 'cupper_points',
 'total_cup_points',
 'moisture',
 'unit_of_measurement',
 'altitude_mean_meters',
 'harvest_year']

In [4]:
# Clean year entries
df_cqi_wrangle['harvest_year'] = df_cqi_wrangle['harvest_year'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x
)

In [5]:
df_cqi_wrangle['harvest_year'] = pd.to_numeric(df_cqi_wrangle['harvest_year'], errors='coerce')

## 4. Data Wrangle - Origin

In [40]:
df_cqi_wrangle['origin_country'].value_counts(dropna=False)

origin_country
mexico              236
colombia            184
guatemala           181
brazil              132
united states        83
taiwan               75
honduras             53
costa rica           51
ethiopia             44
tanzania             40
uganda               36
thailand             32
nicaragua            26
kenya                25
el salvador          21
indonesia            20
china                16
india                14
malawi               11
peru                 10
vietnam               8
myanmar               8
haiti                 6
philippines           5
panama                4
puerto rico           4
ecuador               3
laos                  3
burundi               2
papua new guinea      1
japan                 1
zambia                1
rwanda                1
mauritius             1
côte d'ivoire         1
Name: count, dtype: int64

Looks good, no need for wrangling.

In [41]:
# Check unique values
df_cqi_wrangle['region'].value_counts(dropna=False)

region
huila                            112
oriente                           80
south of minas                    68
kona                              66
unknown                           59
                                ... 
huautla de jimenez                 1
chocaman, veracruz                 1
sierra alta mixe y zapoteca        1
marmelade                          1
kwanza norte province, angola      1
Name: count, Length: 357, dtype: int64

In [55]:
# Build lowercase country list
known_countries = df_cqi_wrangle['origin_country'].dropna().unique().tolist()

In [56]:
# Create flags
df_cqi_wrangle['region_contains_country'] = df_cqi_wrangle['region'].apply(
    lambda x: any(country in str(x).lower() for country in known_countries)
)

In [57]:
# Create flags
df_cqi_wrangle['region_contains_country'] = df_cqi_wrangle['region'].apply(
    lambda x: any(country in str(x).lower() for country in known_countries)
)

df_cqi_wrangle['region_has_delimiter'] = df_cqi_wrangle['region'].str.contains(r'[,/-]', na=False)

df_cqi_wrangle['region_word_count'] = df_cqi_wrangle['region'].str.split().str.len()

df_cqi_wrangle['region_is_unknown'] = df_cqi_wrangle['region'].apply(
    lambda x: str(x).strip().lower() in ['unknown', 'n/a', ''] or pd.isna(x)
)

In [58]:
# Work check
df_cqi_wrangle[
    df_cqi_wrangle[['region_contains_country', 'region_has_delimiter', 'region_is_unknown']].any(axis=1)
][['origin_country', 'region']]

,origin_country,region
0,ethiopia,guji-hambela
1,ethiopia,guji-hambela
2,guatemala,unknown
4,ethiopia,guji-hambela
5,brazil,unknown
...,...,...
1334,ecuador,"san juan, playas"
1335,ecuador,"san juan, playas"
1336,united states,"kwanza norte province, angola"
1337,india,unknown


In [50]:
country_list = df_cqi_wrangle['origin_country'].dropna().unique().tolist()

In [51]:
def remove_country_from_region(region, countries):
    region_lower = str(region).lower()
    for country in countries:
        if country in region_lower:
            region_lower = region_lower.replace(country, '').strip()
    return region_lower if region_lower else 'unknown'

df_cqi_wrangle['region_cleaned'] = df_cqi_wrangle['region'].apply(lambda x: remove_country_from_region(x, country_list))

In [53]:
# Work check
df_cqi_wrangle[['origin_country', 'region', 'region_cleaned']].sample(10, random_state=42)

,origin_country,region,region_cleaned
394,thailand,unknown,unknown
881,indonesia,bener meriah,bener meriah
358,indonesia,bener meriah,bener meriah
367,mexico,xalapa,xalapa
259,mexico,veracruz,veracruz
874,tanzania,mbinga,mbinga
1202,mexico,zaragoza itundujia,zaragoza itundujia
1220,united states,kona,kona
900,malawi,mzuzu,mzuzu
710,costa rica,tarrazu,tarrazu


In [54]:
# Work check where changes were made
df_cqi_wrangle[df_cqi_wrangle['region'] != df_cqi_wrangle['region_cleaned']][['origin_country', 'region', 'region_cleaned']]

,origin_country,region,region_cleaned
46,uganda,eastern uganda,eastern
79,kenya,central kenya,central
80,kenya,central kenya,central
88,ethiopia,"kefa zone, gimbo distict, at a place called wo...","kefa zone, gimbo distict, at a place called wo..."
96,uganda,eastern uganda,eastern
101,kenya,central kenya,central
103,mexico,mexico,unknown
120,thailand,thailand,unknown
126,indonesia,"temanggung, indonesia","temanggung,"
127,japan,ada okinawa japan,ada okinawa


In [59]:
# Clean up the ',' in the region_cleaned
df_cqi_wrangle['region_cleaned'] = df_cqi_wrangle['region_cleaned'].str.strip(" ,")

In [60]:
# Work check for the ','
df_cqi_wrangle[df_cqi_wrangle['region'] != df_cqi_wrangle['region_cleaned']][['origin_country', 'region', 'region_cleaned']]

,origin_country,region,region_cleaned
46,uganda,eastern uganda,eastern
79,kenya,central kenya,central
80,kenya,central kenya,central
88,ethiopia,"kefa zone, gimbo distict, at a place called wo...","kefa zone, gimbo distict, at a place called wo..."
96,uganda,eastern uganda,eastern
101,kenya,central kenya,central
103,mexico,mexico,unknown
120,thailand,thailand,unknown
126,indonesia,"temanggung, indonesia",temanggung
127,japan,ada okinawa japan,ada okinawa


In [61]:
# Investigate weird row spelling
df_cqi_wrangle[df_cqi_wrangle['region'].str.contains('kefa zone', case=False, na=False)]

,species,owner,origin_country,region,local_partner,variety,processing_method,aroma,flavor,aftertaste,...,total_cup_points,moisture,unit_of_measurement,altitude_mean_meters,harvest_year,region_contains_country,region_has_delimiter,region_word_count,region_is_unknown,region_cleaned
88,arabica,seid damtew coffee planataion,ethiopia,"kefa zone, gimbo distict, at a place called wo...",metad agricultural development plc,other,natural / dry,7.75,8.0,7.58,...,85.08,0.11,m,NaN,[2016],True,True,13,False,"kefa zone, gimbo distict, at a place called wo..."


In [62]:
# Cleaning the region entry
df_cqi_wrangle.loc[88, 'region_cleaned'] = "gimbo district, kefa zone"

In [63]:
# Work check
df_cqi_wrangle[df_cqi_wrangle['region'].str.contains('kefa zone', case=False, na=False)]

,species,owner,origin_country,region,local_partner,variety,processing_method,aroma,flavor,aftertaste,...,total_cup_points,moisture,unit_of_measurement,altitude_mean_meters,harvest_year,region_contains_country,region_has_delimiter,region_word_count,region_is_unknown,region_cleaned
88,arabica,seid damtew coffee planataion,ethiopia,"kefa zone, gimbo distict, at a place called wo...",metad agricultural development plc,other,natural / dry,7.75,8.0,7.58,...,85.08,0.11,m,NaN,[2016],True,True,13,False,"gimbo district, kefa zone"


In [64]:
# Drop unneeded columns
df_cqi_wrangle.drop(columns=[
    'region_contains_country',
    'region_has_delimiter',
    'region_word_count',
    'region_is_unknown',
    'region'
], inplace=True)

In [65]:
# Rename region_cleaned to region
df_cqi_wrangle.rename(columns={'region_cleaned': 'region'}, inplace=True)

In [66]:
# Work check
df_cqi_wrangle.columns

Index(['species', 'owner', 'origin_country', 'local_partner', 'variety',
       'processing_method', 'aroma', 'flavor', 'aftertaste', 'acidity', 'body',
       'balance', 'uniformity', 'clean_cup', 'sweetness', 'cupper_points',
       'total_cup_points', 'moisture', 'unit_of_measurement',
       'altitude_mean_meters', 'harvest_year', 'region'],
      dtype='object')

## 5. Reorganized the Columns

In [67]:
column_order = [
    # Entity and origin
    'species',
    'owner',
    'origin_country',
    'region',
    'local_partner',

    # Coffee details
    'variety',
    'processing_method',

    # Sensory evaluation
    'aroma',
    'flavor',
    'aftertaste',
    'acidity',
    'body',
    'balance',
    'uniformity',
    'clean_cup',
    'sweetness',
    'cupper_points',
    'total_cup_points',

    # Physical metrics
    'moisture',
    'unit_of_measurement',
    'altitude_mean_meters',

    # Temporal context
    'harvest_year'
]

In [69]:
# Work check
df_cqi_wrangle.head()

,species,owner,origin_country,local_partner,variety,processing_method,aroma,flavor,aftertaste,acidity,...,uniformity,clean_cup,sweetness,cupper_points,total_cup_points,moisture,unit_of_measurement,altitude_mean_meters,harvest_year,region
0,arabica,metad plc,ethiopia,metad agricultural development plc,unknown,washed / wet,8.67,8.83,8.67,8.75,...,10.0,10.0,10.0,8.75,90.58,0.12,m,2075.0,[2014],guji-hambela
1,arabica,metad plc,ethiopia,metad agricultural development plc,other,washed / wet,8.75,8.67,8.50,8.58,...,10.0,10.0,10.0,8.58,89.92,0.12,m,2075.0,[2014],guji-hambela
2,arabica,grounds for health admin,guatemala,specialty coffee association,bourbon,unknown,8.42,8.50,8.42,8.42,...,10.0,10.0,10.0,9.25,89.75,0.00,m,1700.0,NaN,unknown
3,arabica,yidnekachew dabessa,ethiopia,metad agricultural development plc,unknown,natural / dry,8.17,8.58,8.42,8.42,...,10.0,10.0,10.0,8.67,89.00,0.11,m,2000.0,[2014],oromia
4,arabica,metad plc,ethiopia,metad agricultural development plc,other,washed / wet,8.25,8.50,8.25,8.50,...,10.0,10.0,10.0,8.58,88.83,0.12,m,2075.0,[2014],guji-hambela


## 6. Export

In [7]:
# Export
df_cqi_wrangle.to_csv(
    r'C:\Users\Chase\anaconda_projects\Exercise_6_Coffee\A6_Coffee\02_Data\Prepared_Data\Coffee_Quality_database_from_CQI\coffee_quality_database_from_cqi_wrangled.csv',
    index=False
)